# Notebook 03: Fine-Tuning Runs

**Goal:** Fine-tune on Python code (primary) and TinyStories prose (mandatory control).

**Outputs:** `checkpoints/code_seed*/`, `checkpoints/prose_seed*/`

**Runtime:** ~80 minutes per condition on Colab T4 GPU.

> **Note:** Both conditions are required. The prose control is mandatory for any domain-shift claim.

In [ ]:
import os
import sys
import numpy as np, torch, matplotlib.pyplot as plt
from pathlib import Path

# Define repository information
repo_name = "Mechanistic-Interpretability-Study-Induction-Circuit-Stability-Under-Fine-Tuning"
repo_url = "https://github.com/Mattral/Mechanistic-Interpretability-Study-Induction-Circuit-Stability-Under-Fine-Tuning"
repo_path = f"/content/{repo_name}"  # Standard Colab clone location

# Clone the repository if it doesn't exist
if not os.path.exists(repo_path):
    print(f"Cloning {repo_url} to {repo_path}...")
    !git clone {repo_url} {repo_path}

# Change current working directory to the repository root
# This allows relative imports (like 'src.model...') to work correctly
if os.getcwd() != repo_path:
    print(f"Changing current directory to {repo_path}")
    os.chdir(repo_path)

# Add the current directory (repo root) to sys.path if not already there
# This ensures 'src' is discoverable for imports.
if '.' not in sys.path:
    sys.path.insert(0, '.')

# Install project dependencies from requirements.txt
# This ensures all necessary libraries, including transformer_lens and transformers,
# are installed with the versions specified by the project.
print(f"Installing dependencies from {repo_path}/requirements.txt...")
!pip install -r requirements.txt


### ⚠️ Restart Runtime Required
Please go to **Runtime -> Restart session** now. After the session restarts, run the cell below to load the modules and continue.

In [1]:
# After restarting the runtime, re-run this cell to continue with the imports and model setup.
import numpy as np, torch, matplotlib.pyplot as plt
from pathlib import Path

# The current working directory should already be set to the repo root from the previous cell.
# Add the current directory (repo root) to sys.path if not already there, in case of a fresh restart.
import sys
import os

repo_name = "Mechanistic-Interpretability-Study-Induction-Circuit-Stability-Under-Fine-Tuning"
repo_path = f"/content/{repo_name}"

if os.getcwd() != repo_path:
    print(f"Changing current directory to {repo_path}")
    os.chdir(repo_path)

if '.' not in sys.path:
    sys.path.insert(0, '.')

from src.model.config import ModelConfig, EvalConfig
from src.model.train import load_pretrained_model, set_global_seed
from src.circuits.patching import compute_circuit_attribution, get_circuit_heads
from src.viz.circuit_diagram import plot_circuit_diagram, plot_attribution_heatmap

set_global_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Changing current directory to /content/Mechanistic-Interpretability-Study-Induction-Circuit-Stability-Under-Fine-Tuning
Using device: cuda


In [2]:
import sys; sys.path.insert(0, '..')
import torch
from src.model.config import ModelConfig, TrainConfig
from src.model.finetune import run_finetuning
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
model_config = ModelConfig()

Device: cuda


In [3]:
!grep -n "prepend_bos" /content/Mechanistic-Interpretability-Study-Induction-Circuit-Stability-Under-Fine-Tuning/src/circuits/induction_score.py

22:invoked for this code path; prepend_bos has no effect here. We still pass
23:prepend_bos=False defensively and validate the cache shape is exactly
88:            prepend_bos=False,
103:                "and prepend_bos behaviour for the input type used."
185:            prepend_bos=False,


In [7]:
# --- Code fine-tuning (primary condition, 3 seeds) ---
for seed in [42, 123, 7]:
    print(f'\n=== Code, seed={seed} ===')
    cfg = TrainConfig(seed=seed, checkpoint_dir='../checkpoints', results_dir='../experiments/results')
    history = run_finetuning(model_config, cfg, run_name=f'code_seed{seed}', device=device, prose_control=False)
    print(f'Done: {len(history)} checkpoints')

2026-06-13T11:15:56 | INFO     | src.model.train | Global seed set to 42
2026-06-13T11:15:56 | INFO     | src.model.finetune | Fine-tuning run: code_seed42 | device: cuda
2026-06-13T11:15:56 | INFO     | src.model.train | Loading model 'attn-only-2l' on device 'cuda'



=== Code, seed=42 ===


2026-06-13T11:15:58 | INFO     | src.model.train | Model loaded: 2L 8H d_model=512


Loaded pretrained model attn-only-2l into HookedTransformer
Moving model to device:  cuda


2026-06-13T11:15:58 | INFO     | src.model.finetune | Baseline induction score mean: 0.0297
2026-06-13T11:15:59 | INFO     | src.circuits.patching | Circuit heads (>=0.5): [(1, 6)]
2026-06-13T11:15:59 | INFO     | src.model.finetune | Step-0 | IS_mean=0.0297 | task_loss=11.9075 | logit_diff_clean=4.6309
2026-06-13T11:16:00 | INFO     | src.model.train | Checkpoint saved step=0 -> ../checkpoints/code_seed42/step_000000.pt
2026-06-13T11:16:00 | INFO     | src.model.train | Global seed set to 42
2026-06-13T11:16:00 | INFO     | src.model.train | Training: 244 total steps, checkpoint every 100


...

Loaded pretrained model attn-only-2l into HookedTransformer
Moving model to device:  cuda


2026-06-13T11:18:25 | INFO     | src.model.finetune | Baseline induction score mean: 0.0299
2026-06-13T11:18:26 | INFO     | src.circuits.patching | Circuit heads (>=0.5): [(1, 6)]
2026-06-13T11:18:26 | INFO     | src.model.finetune | Step-0 | IS_mean=0.0299 | task_loss=11.5897 | logit_diff_clean=5.0795
2026-06-13T11:18:26 | INFO     | src.model.train | Checkpoint saved step=0 -> ../checkpoints/code_seed7/step_000000.pt
2026-06-13T11:18:26 | INFO     | src.model.train | Global seed set to 7
2026-06-13T11:18:26 | INFO     | src.model.train | Training: 244 total steps, checkpoint every 100


Moving model to device:  cuda


2026-06-13T11:18:31 | INFO     | src.model.train | step=10 | loss=7.1538 | lr=1.67e-05 | tokens=20480
2026-06-13T11:18:33 | INFO     | src.model.train | step=20 | loss=6.1292 | lr=1.99e-05 | tokens=40960
2026-06-13T11:18:34 | INFO     | src.model.train | step=30 | loss=5.0706 | lr=1.97e-05 | tokens=61440
2026-06-13T11:18:36 | INFO     | src.model.train | step=40 | loss=5.7579 | lr=1.93e-05 | tokens=81920
2026-06-13T11:18:37 | INFO     | src.model.train | step=50 | loss=4.9578 | lr=1.87e-05 | tokens=102400
2026-06-13T11:18:39 | INFO     | src.model.train | step=60 | loss=5.4349 | lr=1.80e-05 | tokens=122880
2026-06-13T11:18:40 | INFO     | src.model.train | step=70 | loss=4.8465 | lr=1.71e-05 | tokens=143360
2026-06-13T11:18:42 | INFO     | src.model.train | step=80 | loss=2.8663 | lr=1.61e-05 | tokens=163840
2026-06-13T11:18:44 | INFO     | src.model.train | step=90 | loss=4.0608 | lr=1.49e-05 | tokens=184320
2026-06-13T11:18:45 | INFO     | src.model.train | step=100 | loss=3.8556 | l

Done: 2 checkpoints


In [10]:
# Create a zip archive of the checkpoints and results directories.
# The `zip -r` command recursively zips the contents of the specified directories.
# We are zipping the directories from `/content/` since they were generated there.
print("Creating zip archive of checkpoints and results...")
!zip -r /content/finetuning_outputs.zip /content/checkpoints /content/experiments/results

print("\nZip file created: /content/finetuning_outputs.zip")
print("You can now download this file from the Colab file browser (usually on the left sidebar) or by running a direct download command if you prefer (e.g., `from google.colab import files; files.download('/content/finetuning_outputs.zip')`).")

Creating zip archive of checkpoints and results...
  adding: content/checkpoints/ (stored 0%)
  adding: content/checkpoints/code_seed123/ (stored 0%)
  adding: content/checkpoints/code_seed123/step_000100.pt (deflated 32%)
  adding: content/checkpoints/code_seed123/step_000000.pt (deflated 8%)
  adding: content/checkpoints/code_seed123/step_000200.pt (deflated 30%)
  adding: content/checkpoints/code_seed7/ (stored 0%)
  adding: content/checkpoints/code_seed7/step_000100.pt (deflated 31%)
  adding: content/checkpoints/code_seed7/step_000000.pt (deflated 8%)
  adding: content/checkpoints/code_seed7/step_000200.pt (deflated 29%)
  adding: content/checkpoints/prose_seed42/ (stored 0%)
  adding: content/checkpoints/prose_seed42/step_000000.pt (deflated 8%)
  adding: content/checkpoints/code_seed42/ (stored 0%)
  adding: content/checkpoints/code_seed42/step_000100.pt (deflated 32%)
  adding: content/checkpoints/code_seed42/step_000000.pt (deflated 8%)
  adding: content/checkpoints/code_seed4

In [11]:
# --- Prose control (mandatory, 3 seeds) ---
for seed in [42, 123, 7]:
    print(f'\n=== Prose, seed={seed} ===')
    cfg = TrainConfig(seed=seed, checkpoint_dir='../checkpoints', results_dir='../experiments/results')
    history = run_finetuning(model_config, cfg, run_name=f'prose_seed{seed}', device=device, prose_control=True)
    print(f'Done: {len(history)} checkpoints')

2026-06-13T11:28:31 | INFO     | src.model.train | Global seed set to 42
2026-06-13T11:28:31 | INFO     | src.model.finetune | Fine-tuning run: prose_seed42 | device: cuda
2026-06-13T11:28:31 | INFO     | src.model.train | Loading model 'attn-only-2l' on device 'cuda'



=== Prose, seed=42 ===


2026-06-13T11:28:33 | INFO     | src.model.train | Model loaded: 2L 8H d_model=512


Loaded pretrained model attn-only-2l into HookedTransformer
Moving model to device:  cuda


2026-06-13T11:28:33 | INFO     | src.model.finetune | Baseline induction score mean: 0.0297
2026-06-13T11:28:34 | INFO     | src.circuits.patching | Circuit heads (>=0.5): [(1, 6)]
2026-06-13T11:28:34 | INFO     | src.model.finetune | Step-0 | IS_mean=0.0297 | task_loss=11.9075 | logit_diff_clean=4.6309
2026-06-13T11:28:35 | INFO     | src.model.train | Checkpoint saved step=0 -> ../checkpoints/prose_seed42/step_000000.pt
2026-06-13T11:28:35 | INFO     | src.model.train | Global seed set to 42
2026-06-13T11:28:35 | INFO     | src.model.train | Training: 244 total steps, checkpoint every 100


...

Loaded pretrained model attn-only-2l into HookedTransformer
Moving model to device:  cuda


2026-06-13T11:31:02 | INFO     | src.model.finetune | Baseline induction score mean: 0.0299
2026-06-13T11:31:03 | INFO     | src.circuits.patching | Circuit heads (>=0.5): [(1, 6)]
2026-06-13T11:31:03 | INFO     | src.model.finetune | Step-0 | IS_mean=0.0299 | task_loss=11.5897 | logit_diff_clean=5.0795
2026-06-13T11:31:04 | INFO     | src.model.train | Checkpoint saved step=0 -> ../checkpoints/prose_seed7/step_000000.pt
2026-06-13T11:31:04 | INFO     | src.model.train | Global seed set to 7
2026-06-13T11:31:04 | INFO     | src.model.train | Training: 244 total steps, checkpoint every 100


Moving model to device:  cuda


2026-06-13T11:31:07 | INFO     | src.model.train | step=10 | loss=6.9265 | lr=1.67e-05 | tokens=20480
2026-06-13T11:31:08 | INFO     | src.model.train | step=20 | loss=5.7886 | lr=1.99e-05 | tokens=40960
2026-06-13T11:31:10 | INFO     | src.model.train | step=30 | loss=5.4699 | lr=1.97e-05 | tokens=61440
2026-06-13T11:31:11 | INFO     | src.model.train | step=40 | loss=5.1979 | lr=1.93e-05 | tokens=81920
2026-06-13T11:31:13 | INFO     | src.model.train | step=50 | loss=4.8301 | lr=1.87e-05 | tokens=102400
2026-06-13T11:31:14 | INFO     | src.model.train | step=60 | loss=4.7100 | lr=1.80e-05 | tokens=122880
2026-06-13T11:31:16 | INFO     | src.model.train | step=70 | loss=4.3636 | lr=1.71e-05 | tokens=143360
2026-06-13T11:31:17 | INFO     | src.model.train | step=80 | loss=4.3472 | lr=1.61e-05 | tokens=163840
2026-06-13T11:31:19 | INFO     | src.model.train | step=90 | loss=4.1157 | lr=1.49e-05 | tokens=184320
2026-06-13T11:31:20 | INFO     | src.model.train | step=100 | loss=4.0954 | l

Done: 2 checkpoints


In [12]:
# Check the total size of the checkpoint_dir and results_dir
# Use 'du -sh' to display disk usage in a human-readable format for the specified directories.
print("Size of checkpoints directory:")
!du -sh /content/checkpoints

print("\nSize of experiments/results directory:")
!du -sh /content/experiments/results

Size of checkpoints directory:
8.2G	/content/checkpoints

Size of experiments/results directory:
28K	/content/experiments/results


In [17]:
# Create a zip archive containing only the experiments/results directory
print("Creating zip archive of experiments/results...")
!zip -r /content/results_only.zip /content/experiments/results

print("\nZip file created: /content/results_only.zip")
print("You can now download this file from the Colab file browser or using the code below.")

Creating zip archive of experiments/results...
updating: content/experiments/results/ (stored 0%)
updating: content/experiments/results/code_seed42_history.npz (deflated 68%)
updating: content/experiments/results/code_seed123_history.npz (deflated 68%)
updating: content/experiments/results/prose_seed42_history.npz (deflated 68%)
updating: content/experiments/results/prose_seed7_history.npz (deflated 68%)
updating: content/experiments/results/prose_seed123_history.npz (deflated 68%)
updating: content/experiments/results/code_seed7_history.npz (deflated 68%)

Zip file created: /content/results_only.zip
You can now download this file from the Colab file browser or using the code below.


In [18]:
from google.colab import files

# Download the zip file containing only the experiments/results
files.download('/content/results_only.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>